# Baseline (cota superior): Qwen3-8B sobre la normativa de pregrado (Ingeniería UdeC)

Réplica del baseline usando **Qwen3-8B**, el mayor modelo permitido (8B), sobre el mismo conjunto de 50 preguntas y con *prompting* directo (sin RAG). Su propósito es comprobar si el modelo de mayor tamaño incurre en las mismas fallas que Qwen3-4B, de ser así, se evidencia que la falla es estructural (ausencia del conocimiento en los pesos) y no un efecto del tamaño del modelo.

El modelo se carga cuantizado a 4 bits para operar dentro de la memoria de una GPU T4 gratuita.


## 1. Verificación de GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No hay GPU disponible."
print(torch.cuda.get_device_name(0),
      "-", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")

Tesla T4 - 15.6 GB


## 2. Dependencias

In [ ]:
!pip install -q -U "transformers>=4.51.0" accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 77.6 MB/s eta 0:00:00


## 3. Carga del modelo (Qwen3-8B, Apache 2.0, cuantización de 4 bits)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_NAME = "Qwen/Qwen3-8B"
bnb = BitsAndBytesConfig(load_in_4bit=True,
                         bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb, device_map="auto")
print(MODEL_NAME, "|", round(torch.cuda.memory_allocated()/1e9, 2), "GB VRAM")

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Qwen/Qwen3-8B | 6.41 GB VRAM


## 4. Configuración de inferencia

Idéntica a la del baseline principal: *prompting* directo, sin documentos, decodificación determinista; se solicita el dato y la cita del artículo, con instrucción de abstención.

In [ ]:
SYSTEM = (
    "Eres un asistente experto en la normativa de pregrado de la Facultad de "
    "Ingeniería de la Universidad de Concepción. Responde de forma breve, con el "
    "dato exacto y citando el artículo correspondiente (por ejemplo: 'Art. 8'). "
    "Si no tienes la información, di explícitamente que no está en la normativa."
)

def preguntar(pregunta, sistema=SYSTEM, max_new_tokens=256):
    messages = [{"role": "system", "content": sistema},
                {"role": "user", "content": pregunta}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True).strip()

## 5. Conjunto de evaluación

El mismo conjunto de 50 preguntas (10 por categoría) usado en el baseline principal. RG = Reglamento General; RI-FI = Reglamento Interno de Ingeniería; CAL = Calendario de Docencia 2026.

In [ ]:
preguntas = [
    {"categoria": "factual", "pregunta": "¿Cuál es la nota mínima para aprobar una asignatura en la Facultad de Ingeniería?", "gold_dato": "4,0", "gold_fuente": "Art. 11, RI-FI"},
    {"categoria": "factual", "pregunta": "¿Cuántas evaluaciones sumativas como mínimo debe tener una asignatura?", "gold_dato": "al menos 3 (tres)", "gold_fuente": "Art. 11, RI-FI"},
    {"categoria": "factual", "pregunta": "¿A cuántas evaluaciones de recuperación tiene derecho el estudiante por asignatura?", "gold_dato": "una (1)", "gold_fuente": "Art. 12, RI-FI"},
    {"categoria": "factual", "pregunta": "¿En qué escala se expresa la nota final de una asignatura?", "gold_dato": "escala oficial de 1 a 7, con un decimal", "gold_fuente": "Art. 23, RG"},
    {"categoria": "factual", "pregunta": "¿Cuál es la actividad final de titulación en las carreras de Ingeniería?", "gold_dato": "la Memoria de Título", "gold_fuente": "Art. 29, RI-FI"},
    {"categoria": "factual", "pregunta": "¿Con cuánta anticipación mínima deben comunicarse las evaluaciones a los estudiantes?", "gold_dato": "al menos una semana", "gold_fuente": "Art. 10, RI-FI"},
    {"categoria": "factual", "pregunta": "¿Quién resuelve en primera instancia una solicitud de continuación de estudios?", "gold_dato": "el Comité de Docencia y Asuntos Estudiantiles", "gold_fuente": "Art. 16, RI-FI"},
    {"categoria": "factual", "pregunta": "¿La renuncia a la carrera es revocable?", "gold_dato": "No, es irrevocable", "gold_fuente": "Art. 26, RI-FI"},
    {"categoria": "factual", "pregunta": "¿Cuántos períodos lectivos ordinarios tiene el año académico y cuánto duran?", "gold_dato": "dos, de 19 semanas cada uno", "gold_fuente": "Art. 16, RG"},
    {"categoria": "factual", "pregunta": "¿Los estudiantes de Primer Año Común pueden convalidar asignaturas?", "gold_dato": "No", "gold_fuente": "Art. 22, RI-FI"},
    {"categoria": "numerica", "pregunta": "¿Cuál es el mínimo de créditos que debo inscribir por período lectivo ordinario en Ingeniería?", "gold_dato": "8 créditos (igual o superior a 8)", "gold_fuente": "Art. 8, RI-FI"},
    {"categoria": "numerica", "pregunta": "¿Con menos de cuántos créditos aprobados al término del segundo semestre quedo en baja académica?", "gold_dato": "menos de 15 créditos", "gold_fuente": "Art. 14 a), RI-FI"},
    {"categoria": "numerica", "pregunta": "¿Bajo qué promedio de créditos aprobados por semestre (desde el 4º) se cae en baja académica?", "gold_dato": "inferior a 10 créditos semestrales", "gold_fuente": "Art. 14 b), RI-FI"},
    {"categoria": "numerica", "pregunta": "¿Cuál es la asistencia mínima máxima que un profesor puede exigir en clases teóricas y prácticas?", "gold_dato": "80%", "gold_fuente": "Art. 13, RI-FI"},
    {"categoria": "numerica", "pregunta": "¿Qué calificación mínima se exige para convalidar una asignatura?", "gold_dato": "4,5", "gold_fuente": "Art. 21, RI-FI"},
    {"categoria": "numerica", "pregunta": "¿Durante cuántas semanas iniciales se puede solicitar convalidación, revalidación o reconocimiento?", "gold_dato": "las 4 primeras semanas", "gold_fuente": "Art. 22, RI-FI"},
    {"categoria": "numerica", "pregunta": "¿Dentro de cuántos días se puede solicitar el reconocimiento de asignaturas?", "gold_dato": "los primeros 30 días", "gold_fuente": "Art. 23, RI-FI"},
    {"categoria": "numerica", "pregunta": "Si suspendí más de 3 años y solo me faltaba la Memoria, ¿cuántos créditos adicionales máximo debo cursar?", "gold_dato": "un máximo de 36 créditos", "gold_fuente": "Art. 31, RI-FI"},
    {"categoria": "numerica", "pregunta": "¿Cuántas semanas dura cada período lectivo ordinario (semestre)?", "gold_dato": "19 semanas", "gold_fuente": "Art. 16, RG"},
    {"categoria": "numerica", "pregunta": "¿Cuál es el plazo (en días hábiles) para regularizar una evaluación no rendida por causa justificada?", "gold_dato": "3 días hábiles", "gold_fuente": "Art. 26, RG"},
    {"categoria": "condicional", "pregunta": "Si vengo de otra universidad y quiero ingresar a Ingeniería, ¿qué promedio necesito? ¿Y para cambio dentro de Ingeniería?", "gold_dato": "4,5 desde otra universidad/facultad; 4,2 para cambio dentro de Ingeniería", "gold_fuente": "Art. 17 y Art. 18, RI-FI"},
    {"categoria": "condicional", "pregunta": "¿Qué promedio mínimo se exige para cursar una segunda carrera de forma simultánea?", "gold_dato": "5,0", "gold_fuente": "Art. 19, RI-FI"},
    {"categoria": "condicional", "pregunta": "¿En qué caso NO se puede revalidar una asignatura?", "gold_dato": "si fue previamente cursada y reprobada", "gold_fuente": "Art. 20, RI-FI"},
    {"categoria": "condicional", "pregunta": "Si repruebo por segunda vez una misma asignatura obligatoria, ¿qué ocurre?", "gold_dato": "baja académica, salvo que la haya aprobado en el PLEV o esté en primer año", "gold_fuente": "Art. 14 c), RI-FI"},
    {"categoria": "condicional", "pregunta": "Al modificar la inscripción, ¿qué asignaturas NO puedo eliminar?", "gold_dato": "las atrasadas y las obligatorias reprobadas (prioridades a y b del Art. 7)", "gold_fuente": "Art. 9, RI-FI"},
    {"categoria": "condicional", "pregunta": "Si suspendí mis estudios, ¿qué debo hacer para volver?", "gold_dato": "solicitar la reincorporación a la Vicedecanatura", "gold_fuente": "Art. 27, RI-FI"},
    {"categoria": "condicional", "pregunta": "¿Qué documentos requiere la solicitud de suspensión de estudios?", "gold_dato": "certificado de la DAFE (sin deudas) y de la Dirección de Bibliotecas", "gold_fuente": "Art. 25, RI-FI"},
    {"categoria": "condicional", "pregunta": "Si estando habilitado no inscribo asignaturas en el período, ¿qué pasa?", "gold_dato": "pierdo el derecho a continuar y la calidad de alumno (Baja por no Inscripción)", "gold_fuente": "Art. 29, RG"},
    {"categoria": "condicional", "pregunta": "Si no me presento a una evaluación por causa justificada, ¿qué puedo hacer?", "gold_dato": "solicitar regularizar en un máximo de 3 días hábiles", "gold_fuente": "Art. 26, RG"},
    {"categoria": "condicional", "pregunta": "¿Cuánta asistencia puede exigir un profesor en un laboratorio, taller o salida a terreno?", "gold_dato": "hasta un 100%", "gold_fuente": "Art. 13, RI-FI"},
    {"categoria": "cruce", "pregunta": "Si el Reglamento Interno de Ingeniería contradice al Reglamento General de Docencia, ¿cuál prevalece?", "gold_dato": "prevalece el Reglamento General (y las Normas de Ingreso)", "gold_fuente": "Art. 34, RI-FI"},
    {"categoria": "cruce", "pregunta": "Quiero modificar las asignaturas inscritas este segundo semestre 2026. ¿Hasta qué fecha?", "gold_dato": "hasta el 04 de septiembre de 2026", "gold_fuente": "CAL 2º sem 2026 (Art. 9, RI-FI)"},
    {"categoria": "cruce", "pregunta": "¿Hasta qué fecha se pueden modificar asignaturas en el primer semestre 2026?", "gold_dato": "hasta el 02 de abril de 2026", "gold_fuente": "CAL 1er sem 2026 (Art. 9, RI-FI)"},
    {"categoria": "cruce", "pregunta": "¿Tengo derecho a suspender estudios y hasta cuándo puedo hacerlo?", "gold_dato": "sí; dentro de los plazos del calendario, a más tardar 4 semanas antes del término", "gold_fuente": "Art. 25 RI-FI + Art. 31 RG"},
    {"categoria": "cruce", "pregunta": "¿Cuándo inician las clases del segundo semestre 2026?", "gold_dato": "10 de agosto de 2026", "gold_fuente": "CAL 2º sem 2026"},
    {"categoria": "cruce", "pregunta": "¿En qué fechas se inscriben las asignaturas del segundo semestre 2026 y cuál es el mínimo de créditos?", "gold_dato": "inscripción 03–07 de agosto de 2026; mínimo 8 créditos", "gold_fuente": "CAL 2º sem 2026 + Art. 8, RI-FI"},
    {"categoria": "cruce", "pregunta": "¿Cuándo termina el segundo semestre 2026 (término de clases)?", "gold_dato": "11 de diciembre de 2026", "gold_fuente": "CAL 2º sem 2026"},
    {"categoria": "cruce", "pregunta": "La escala de notas del Reglamento General, ¿se aplica en Ingeniería, y cuál es la nota de aprobación?", "gold_dato": "sí, escala 1 a 7 (Art. 23, RG); se aprueba con 4,0 (Art. 11, RI-FI)", "gold_fuente": "Art. 23 RG + Art. 11 RI-FI"},
    {"categoria": "cruce", "pregunta": "¿Cuándo son las Evaluaciones de Recuperación del segundo semestre 2026?", "gold_dato": "14–23 de diciembre de 2026", "gold_fuente": "CAL 2º sem 2026 (Art. 12 RI-FI)"},
    {"categoria": "cruce", "pregunta": "¿Qué reglamento rige la Memoria de Título y cuánto pondera en la nota de titulación?", "gold_dato": "la rige el Reglamento de Memoria de Título de la FI (Art. 29); pondera un 40% (Art. 30)", "gold_fuente": "Art. 29 y Art. 30, RI-FI"},
    {"categoria": "abstencion", "pregunta": "¿Qué establece el Artículo 90 del Reglamento Interno de Docencia de Pregrado de Ingeniería?", "gold_dato": "El artículo no existe: el RI-FI tiene solo 35 artículos -> debe indicarlo, no inventar", "gold_fuente": "dato inexistente (RI-FI llega al Art. 35)"},
    {"categoria": "abstencion", "pregunta": "¿Qué requisitos fija el Artículo 36 del Reglamento Interno de Docencia de Pregrado de Ingeniería?", "gold_dato": "El artículo no existe: el reglamento termina en el Art. 35 -> debe indicarlo", "gold_fuente": "dato inexistente (RI-FI llega al Art. 35)"},
    {"categoria": "abstencion", "pregunta": "¿Qué dispone el Artículo 100 del Reglamento General de Docencia de Pregrado?", "gold_dato": "El artículo no existe: el RG llega hasta el Art. 60 -> debe indicarlo", "gold_fuente": "dato inexistente (RG llega al Art. 60)"},
    {"categoria": "abstencion", "pregunta": "¿Qué señala el Artículo 70 del Reglamento General de Docencia de Pregrado?", "gold_dato": "El artículo no existe: el RG llega hasta el Art. 60 -> debe indicarlo", "gold_fuente": "dato inexistente (RG llega al Art. 60)"},
    {"categoria": "abstencion", "pregunta": "¿Qué dice el inciso d) del Artículo 14 del RI-FI sobre las causales de baja académica?", "gold_dato": "No existe el inciso d): el Art. 14 solo tiene a), b) y c) -> debe indicarlo", "gold_fuente": "premisa falsa (Art. 14 solo a-c)"},
    {"categoria": "abstencion", "pregunta": "¿Qué establece el inciso c) del Artículo 25 del RI-FI sobre los documentos para suspender estudios?", "gold_dato": "No existe el inciso c): el Art. 25 solo tiene a) y b) -> debe indicarlo", "gold_fuente": "premisa falsa (Art. 25 solo a-b)"},
    {"categoria": "abstencion", "pregunta": "Además del mínimo de créditos, ¿cuál es el número máximo de créditos que fija el Artículo 8 del RI-FI para inscribir por semestre?", "gold_dato": "El Art. 8 no fija un máximo, solo un mínimo (8) -> debe indicarlo", "gold_fuente": "premisa falsa (Art. 8 solo fija mínimo)"},
    {"categoria": "abstencion", "pregunta": "¿Cuál es la nota mínima que fija el Artículo 12 del RI-FI para la Evaluación de Recuperación?", "gold_dato": "El Art. 12 no fija una nota mínima; solo otorga el derecho a la recuperación -> debe indicarlo", "gold_fuente": "premisa falsa (Art. 12 no fija nota)"},
    {"categoria": "abstencion", "pregunta": "Según el calendario de docencia 2026, ¿qué feriado nacional cae el 30 de septiembre?", "gold_dato": "No hay feriado el 30 de septiembre; el feriado de septiembre 2026 es el día 18 -> debe indicarlo", "gold_fuente": "dato inexistente (CAL 2026)"},
    {"categoria": "abstencion", "pregunta": "El Artículo 19 del RI-FI exige promedio 5,0 para una segunda carrera simultánea; ¿qué promedio exige ese artículo para una tercera carrera simultánea?", "gold_dato": "El Art. 19 no contempla una tercera carrera simultánea -> debe indicarlo", "gold_fuente": "premisa falsa (Art. 19 no lo contempla)"},
]

## 6. Ejecución

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

registros = []
for p in preguntas:
    r = preguntar(p["pregunta"])
    registros.append({"categoria": p["categoria"], "pregunta": p["pregunta"],
                      "respuesta_modelo": r, "gold_dato": p["gold_dato"],
                      "gold_fuente": p["gold_fuente"]})

df = pd.DataFrame(registros)
df.to_csv("respuestas_baseline_8b.csv", index=False)
df[["categoria", "respuesta_modelo", "gold_dato", "gold_fuente"]]

,categoria,respuesta_modelo,gold_dato,gold_fuente
0,factual,"La nota mínima para aprobar una asignatura es 4,0. Esto se establece en el **Art. 10** de la Normativa de Pregrado de la Facultad de Ingeniería de la Universidad de Concepción.","4,0","Art. 11, RI-FI"
1,factual,"La normativa no establece un mínimo de evaluaciones sumativas para una asignatura. **Art. 12** indica que las evaluaciones deben ser suficientes para verificar el logro de los aprendizajes, pero no fija un número mínimo.",al menos 3 (tres),"Art. 11, RI-FI"
2,factual,"El estudiante tiene derecho a una evaluación de recuperación por asignatura, según el **Art. 12** de la normativa de pregrado.",una (1),"Art. 12, RI-FI"
3,factual,"La nota final de una asignatura se expresa en escala de 1 a 7, con decimales de 0,5, según el **Art. 10** de la Normativa de Pregrado.","escala oficial de 1 a 7, con un decimal","Art. 23, RG"
4,factual,"La actividad final de titulación en las carreras de Ingeniería es la **Tesis de Graduación**, que debe ser presentada y defendida ante un tribunal calificador, según el **Art. 12** de la Normativa de Pregrado de la Facultad de Ingeniería.",la Memoria de Título,"Art. 29, RI-FI"
5,factual,La normativa no establece un plazo mínimo de anticipación para comunicar las evaluaciones a los estudiantes. Art. 12.,al menos una semana,"Art. 10, RI-FI"
6,factual,El órgano que resuelve en primera instancia una solicitud de continuación de estudios es el **Decano de la Facultad**. \n\nArt. 12.,el Comité de Docencia y Asuntos Estudiantiles,"Art. 16, RI-FI"
7,factual,"No, la renuncia a la carrera no es revocable. Según el **Art. 10** de la Normativa de Pregrado, la renuncia a la carrera implica la pérdida definitiva del derecho a continuar estudiando en dicha carrera.","No, es irrevocable","Art. 26, RI-FI"
8,factual,"El año académico tiene dos períodos lectivos ordinarios, cada uno de 16 semanas de duración. \n\nArt. 10.","dos, de 19 semanas cada uno","Art. 16, RG"
9,factual,"No, los estudiantes de Primer Año Común no pueden convalidar asignaturas. Esto se establece en el **Art. 10** de la Normativa de Pregrado, que establece que el Primer Año Común es un año de formación básica y no se permite la convalidación de asignaturas en este nivel.",No,"Art. 22, RI-FI"


## 7. Calificación y resultados

Calificación estricta (dato y artículo correctos; en premisa falsa, reconocer que el dato no existe).

In [ ]:
veredictos = [
 "no","no","si","no","no","no","no","no","no","no",   # factual (0-9)
 "no","no","no","no","no","no","no","no","no","no",   # numérica (10-19)
 "no","no","no","no","no","no","no","no","no","no",   # condicional (20-29)
 "no","no","no","no","no","no","no","no","no","no",   # cruce (30-39)
 "no","no","no","no","no","no","no","no","no","si",   # abstención (40-49)
]

if len(veredictos) == len(df):
    df["correcto"] = veredictos
    df.to_csv("resultados_baseline_8b.csv", index=False)
    df["acierto"] = df["correcto"].str.strip().str.lower().isin(["si", "sí"])
    resumen = df.groupby("categoria")["acierto"].agg(["sum", "count"])
    resumen["exactitud_%"] = (100*resumen["sum"]/resumen["count"]).round(0)
    print(resumen.reindex(["factual", "numerica", "condicional", "cruce", "abstencion"]))
    print("\nExactitud global: {:.0f}%".format(100*df["acierto"].mean()))
    con = df[df["categoria"] != "abstencion"]
    print("Aciertos en preguntas con respuesta en el corpus: {}/{}".format(
        int(con["acierto"].sum()), len(con)))
else:
    print(f"Calificacion pendiente: completar 'veredictos' con {len(df)} valores (hay {len(veredictos)}).")

             sum  count  exactitud_%
categoria                           
factual        1     10         10.0
numerica       0     10          0.0
condicional    0     10          0.0
cruce          0     10          0.0
abstencion     1     10         10.0

Exactitud global: 4%
Aciertos en preguntas con respuesta en el corpus: 1/40
